# Preprocessing

Extract the required Features and create dataset

## Config

Expand system path to incorporate [src](../src/)

In [ ]:
import sys
sys.path.append('../src')

Load Configuration

In [ ]:
CONFIGURATION = '../configuration'

import configparser
import os
conf = configparser.RawConfigParser()
conf.read((
    os.path.join(CONFIGURATION, f"default.conf"),
    os.path.join(CONFIGURATION, f"{os.uname().nodename}.conf")
))
conf.sections()

# Load metadata

Load tokens

In [ ]:
FILE_TOKENS = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['file-token-list']
)
import pandas as pd
tokens = set(pd.read_pickle(FILE_TOKENS))

print(len(tokens))

Load domains

In [ ]:
DIR_OPENFACE = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['path-openface']
)
DIR_WINDOWS = os.path.join(
    conf['Dataset']['path-root'],
    conf['Dataset']['path-windows']
)
conditions = set(conf['Dataset']['conditions'].split(', '))
assert len(conditions - set(os.listdir(DIR_WINDOWS))) == 0

states = set(conf['Dataset']['states'].split(', '))

assert len(states - {
    state
    for condition in conditions
    for state in os.listdir(os.path.join(DIR_WINDOWS, condition))
}) == 0
conditions, states

Load Action Unit feature selection

In [ ]:
from utils.csv import csv_col_dict
ACTION_UNITS = csv_col_dict(
    conf['Features']['action-units'].strip().split('\n')
)
len(ACTION_UNITS['feature'])

## Create Data Loader

In [ ]:
from dataloader import DataLoaderFull
dl = DataLoaderFull(
    tokens = tokens,
    conditions = conditions,
    states = states,
    source_state = DIR_WINDOWS,
    source_record = DIR_OPENFACE,
    file_record = conf['Dataset']['file-openface']
)

## Create Dataset

Load data and annotate blinks

In [ ]:
data = dl.load_full()
data

Type correction

In [ ]:
data['success'] = data['success'].astype(bool)
for col in data.columns:
    if col.startswith('AU') and col.endswith('_c'):
        data[col] = data[col].astype(bool)

data

Success Filtering

In [ ]:
au_r = [
    col for col in data.columns 
    if col.startswith('AU') and col.endswith('_r')
]
outliers_au = (data[au_r] > 5.0).any(axis=1) != True
data_success = data.copy()
data_success['success'] = data['success'] & outliers_au
data_success

Annotate blinks

In [ ]:
ACTION_UNIT_BLINK = ACTION_UNITS['feature'][ACTION_UNITS['name'].index('AU45')]
TIME_SECONDS = 'timestamp'
from utils.blink import blink_processor
data_blink = blink_processor(data_success, ACTION_UNIT_BLINK, TIME_SECONDS)
data_blink['blink_state'] = data_blink['blink_state'].astype(bool)
data_blink

Select features

In [ ]:
cols = ['success', 'frame','timestamp','token','condition',
        'state_understanding','state_confusion','state_listening',
        'gaze_0_x', 'gaze_0_y', 'gaze_0_z', 'gaze_1_x', 'gaze_1_y', 'gaze_1_z',
        'gaze_angle_x', 'gaze_angle_y', 'pose_Tx', 'pose_Ty', 'pose_Tz',
        'pose_Rx', 'pose_Ry', 'pose_Rz',
        'blink_state', 'blink_period_id', 'blink_period_time', 
        'blink_interval_id', 'blink_interval_time'
    ] + ACTION_UNITS['feature']
selection = data_blink[cols].rename(
    # rename action units (trim _r or _c)
    columns=dict(zip(ACTION_UNITS['feature'], ACTION_UNITS['name']))
).reset_index(drop=True)
selection

Save to file

In [ ]:
FILE_OUT = '../.data/processed'
FORMATS = [
    lambda df, file: df.to_csv(file + '.csv', index=False),
    lambda df, file: df.to_pickle(file + '.pkl'),
]
for save in FORMATS:
    save(selection, FILE_OUT)